In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs"
FILE_NAME = f"1s3w_mamlpp_MOE_HPO_DeepCNNLSTM"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: 1s3w_mamlpp_MOE_HPO_DeepCNNLSTM
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\1s3w_mamlpp_MOE_HPO_DeepCNNLSTM.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'mamlpp_MOE_NumExp_DeepCNNLSTM_2fcv_hpo'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 101 trials.
Best value (Accuracy): 0.9417
Best Hyperparameters:
  cnn_base_filters: 128
  lstm_hidden: 128
  meta_batchsize: 4
  maml_inner_steps: 15
  maml_inner_steps_eval: 100
  maml_alpha_init: 0.002487014323436451
  maml_alpha_init_eval: 0.011690130192128601
  outer_lr: 0.0002767277795088434
  wd: 0.00016686152395082782
  groupnorm_num_groups: 4
  use_GlobalAvgPooling: False
  num_experts: 37
  MOE_top_k: 8
  MOE_placement: encoder
  MOE_gate_temperature: 0.5941228804888874
  MOE_aux_coeff: 0.03370428220209942
  MOE_ctx_out_dim: 32
  MOE_ctx_hidden_dim: 32
  MOE_dropout: 0.06309626206246866
  MOE_aux_loss_plcmt: both
  episodes_per_epoch_train: 400
  label_smooth: 0.2
  use_maml_msl: False
  maml_use_lslr: True
  use_lslr_at_eval: False


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL
params_to_plot = ["cnn_base_filters", "lstm_hidden", "meta_batchsize", "maml_inner_steps", "maml_inner_steps_eval", "maml_alpha_init", "maml_alpha_init_eval", "outer_lr", "wd", "groupnorm_num_groups", "use_GlobalAvgPooling", "num_experts", "MOE_top_k", "MOE_placement", "MOE_gate_temperature", "MOE_aux_coeff", "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout", "MOE_aux_loss_plcmt", "episodes_per_epoch_train", "label_smooth", "use_maml_msl", "maml_use_lslr", "use_lslr_at_eval"]

params_to_plot_ARCH = [
    "cnn_base_filters", "lstm_hidden", "groupnorm_num_groups", "use_GlobalAvgPooling"
]

params_to_plot_OPT = [
    "outer_lr", "wd", "label_smooth", "episodes_per_epoch_train"
]

params_to_plot_MAML = [
    "meta_batchsize", "maml_inner_steps", "maml_inner_steps_eval", "maml_alpha_init", "maml_alpha_init_eval","maml_use_lslr","use_lslr_at_eval","use_maml_msl"
]

params_to_plot_MOE_CORE = [
    "num_experts", "MOE_top_k", "MOE_placement", "MOE_gate_temperature"
]

params_to_plot_MOE_AUX = [
    "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout", "MOE_aux_coeff", "MOE_aux_loss_plcmt"
]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_ARCH)
fig_slice.show()

In [12]:
fig_slice = plot_slice(study, params=params_to_plot_MAML)
fig_slice.show()

In [13]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_CORE)
fig_slice.show()

In [14]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_AUX)
fig_slice.show()

## Best Hyperparameter Ranges

- epochs_per_epoch_train: 200-400 seems fine. Drops off above 500 but maybe just wasnted tested...
- label_smooth: no real trend. 0.15-0.2 maybe a bit better?
- outer_lr: 0.0001-0.001 --> Could maybe go lower?
- wd: > 0.0001
- cnn_base_filters: no real trend... 128 the best ig
- groupnorm_num_groups: no strong trend. 4 used more
- lstm_hidden: 128 most used. 64 okay. 256 and above not great / used as much
- use_GAP: no real trend... False has a tighter spread and was used more
- maml_alpha_init: <0.005
- maml_alpha_init_eval: 0.0075-0.075
- maml_inner_steps: weak trend. Use 15 the most... maybe raise? 5 and 7 were okay... maybe they are actually better idk
- maml_inner_steps_eval: 100 LOL, 50 did fine
- maml_use_lslr: True
- meta_batchsize: 24
- use_lslr_at_eval: False but True looks like it did fine...
- use_maml_msl: False! Hybrid did okay but had higher spread
- MOE_gate_temperature: <1.5
- MOE_placement: encoder
- MOE_top_k: most used 8, 9, 10. 1 was bad. No collapse at 9+
- num_experts: higher better, 20-40. Maybe could raise?
- MOE_aux_coeff: <0.2, lower better. 0.02
- MOE_aux_loss_plcmt: inner > outer, Optuna favored both
- MOE_ctx_hidden_dim: 32. 64 okay but higher variance. 128 trash
- MOE_ctx_out_dim: 32 chosen. No super clear trend
- MOE_dropout: no real trend. Optuna favored ~0.05 towards the end

In [15]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
93,93,0.941667,0.916667,2026-04-05 21:12:17.137878,0 days 02:33:24.910821
54,54,0.940625,0.942708,2026-04-04 23:37:07.509902,0 days 00:37:02.588663
65,65,0.934375,0.937500,2026-04-05 04:10:11.698870,0 days 01:42:17.125079
61,61,0.931250,0.936458,2026-04-05 02:38:00.939986,0 days 01:31:32.370821
74,74,0.927083,0.936458,2026-04-05 09:50:15.753414,0 days 02:56:57.748177
90,90,0.925000,0.916667,2026-04-05 19:11:44.949560,0 days 02:00:20.397616
77,77,0.921875,0.917708,2026-04-05 12:47:24.775890,0 days 01:10:49.419966
92,92,0.917708,0.928125,2026-04-05 20:50:00.839625,0 days 01:20:29.486011
72,72,0.916667,0.923958,2026-04-05 08:30:37.855917,0 days 02:06:15.957043
64,64,0.915625,0.910417,2026-04-05 03:43:51.681449,0 days 00:51:11.390872


In [16]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)

{'cnn_base_filters': 128, 'lstm_hidden': 128, 'meta_batchsize': 4, 'maml_inner_steps': 15, 'maml_inner_steps_eval': 100, 'maml_alpha_init': 0.002487014323436451, 'maml_alpha_init_eval': 0.011690130192128601, 'outer_lr': 0.0002767277795088434, 'wd': 0.00016686152395082782, 'groupnorm_num_groups': 4, 'use_GlobalAvgPooling': False, 'num_experts': 37, 'MOE_top_k': 8, 'MOE_placement': 'encoder', 'MOE_gate_temperature': 0.5941228804888874, 'MOE_aux_coeff': 0.03370428220209942, 'MOE_ctx_out_dim': 32, 'MOE_ctx_hidden_dim': 32, 'MOE_dropout': 0.06309626206246866, 'MOE_aux_loss_plcmt': 'both', 'episodes_per_epoch_train': 400, 'label_smooth': 0.2, 'use_maml_msl': False, 'maml_use_lslr': True, 'use_lslr_at_eval': False}
{'cnn_base_filters': 64, 'lstm_hidden': 64, 'meta_batchsize': 24, 'maml_inner_steps': 7, 'maml_inner_steps_eval': 50, 'maml_alpha_init': 0.0016865261840566302, 'maml_alpha_init_eval': 0.03241222861959444, 'outer_lr': 0.00011753148144028081, 'wd': 0.0006958201039866241, 'groupnorm_n

In [21]:
from optuna.trial import TrialState

# 1. Filter for completed trials with maml_inner_steps_eval == 50
completed_trials = [
    t for t in study.trials
    if t.state == TrialState.COMPLETE and t.params['maml_inner_steps_eval'] == 50
]

# 2. Sort and take top 10
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)

{'cnn_base_filters': 64, 'lstm_hidden': 64, 'meta_batchsize': 24, 'maml_inner_steps': 7, 'maml_inner_steps_eval': 50, 'maml_alpha_init': 0.0016865261840566302, 'maml_alpha_init_eval': 0.03241222861959444, 'outer_lr': 0.00011753148144028081, 'wd': 0.0006958201039866241, 'groupnorm_num_groups': 8, 'use_GlobalAvgPooling': True, 'num_experts': 25, 'MOE_top_k': 6, 'MOE_placement': 'encoder', 'MOE_gate_temperature': 0.5007953923754159, 'MOE_aux_coeff': 0.17418352079333946, 'MOE_ctx_out_dim': 32, 'MOE_ctx_hidden_dim': 64, 'MOE_dropout': 0.10372039932801176, 'MOE_aux_loss_plcmt': 'inner', 'episodes_per_epoch_train': 200, 'label_smooth': 0.05, 'use_maml_msl': 'hybrid', 'maml_msl_num_epochs': 36, 'maml_use_lslr': True, 'use_lslr_at_eval': True}
{'cnn_base_filters': 64, 'lstm_hidden': 128, 'meta_batchsize': 24, 'maml_inner_steps': 7, 'maml_inner_steps_eval': 50, 'maml_alpha_init': 0.003183796421686842, 'maml_alpha_init_eval': 0.02561505243517187, 'outer_lr': 0.00017730562335905792, 'wd': 0.000962